# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min (full: ~2 h on 8×A10G)

---

Previously in Part 3, we turned the training data into tokens. Now we pretrain the foundation model: a transformer that reads the token stream and learns to predict the next token, the same way an LLM learns to predict the next word. The transactions themselves are the training signal — fraud labels play no part in this step.

We start by loading the three files Part 3 wrote, then Ray Train runs the training loop — one CPU worker at `mini`, eight GPUs at `full` — and we finish by saving the trained model to shared storage.

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import logging
import ray
ray.init(ignore_reinit_error=True, logging_level=logging.ERROR,
         runtime_env={"working_dir": DEMO_ROOT,
                      # torch's native JIT wants a C compiler the workers don't have
                      "env_vars": {"TORCH_DISABLE_NATIVE_JIT": "1"}})

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "mini"                       # same knob as the earlier parts; full = 8×A10G, ~2h
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

## Load the training data

Part 3 wrote three files, and this cell loads them for training.

- `ids.npy` — the tokens, one big integer array. Each row is one training example: a 4096-token stretch of one card's history (Part 3 called these windows; at `mini` they are 256 tokens).
- `attn.npy` — a matching 0/1 array. A card's last window is usually not completely full, so the 1s mark the positions that hold real tokens and the model skips the rest.
- `vocab.json` — the token list; the trainer reads it to size the model.

We wrap the two arrays into one Ray dataset. Ray Train splits it across the training workers.

In [2]:
import numpy as np

vocab_path = os.path.join(paths["nvcorpus"], "vocab.json")
ids = np.load(os.path.join(paths["nvcorpus"], "ids.npy"))
attn = np.load(os.path.join(paths["nvcorpus"], "attn.npy"))
n_seqs = int(ids.shape[0])

# Wrap the two arrays as one Ray dataset with the column names the training
# function expects. from_numpy makes a dataset out of an array; zip joins the
# two datasets column-wise.
train_ds = (ray.data.from_numpy(ids).rename_columns({"data": "input_ids"})
            .zip(ray.data.from_numpy(attn).rename_columns({"data": "attention_mask"})))

print(f"{n_seqs:,} windows × {ids.shape[1]} tokens each")
print(f"window 0, first 13 tokens as the model sees them: {ids[0][:13].tolist()}")
print("(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)")

8 windows × 256 tokens each
window 0, first 13 tokens as the model sees them: [1, 8, 679, 2022, 2080, 2142, 2166, 2176, 2179, 2191, 3110, 3198, 3251]
(the same window Part 3 decoded into AMT_3 MERCH_667 CAT_RETAIL ...)


## The model

The model is a transformer from the Llama family — the same architecture behind most open LLMs — at NVIDIA's exact configuration: 29M parameters at `full`, a 2-layer miniature at `mini`. The details live in `src/model.py`. It is a *causal* model: each position sees only the tokens before it, so every prediction uses only the past.

Training is next-token prediction. At every position, the model predicts the token that comes next, and the difference between its guess and the real token is the training signal. Every position in a window is one prediction, so a single 4096-token window carries ~4,000 training examples. Part 5 will use the trained model's internal state as the embedding; this notebook only trains it.

## Train with Ray Train

The training function (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, forward, backward, step, with NVIDIA's recipe for the optimizer and learning-rate schedule coming from the scale config. Ray Train runs that function on every training worker and supplies the distributed pieces: it launches the workers, hands each one a shard of the dataset, averages their gradients after every step so they stay in sync, and writes checkpoints to shared storage.

`ScalingConfig` is the one line that sets the scale: one CPU worker at `mini`, eight GPU workers at `full`, same function either way.

One piece of arithmetic runs before training: the learning-rate schedule needs the total number of optimizer steps up front, so we compute it from the window count, the batch size, the worker count, and the epochs.

In [3]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # the per-worker PyTorch loop
import math

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]

# The schedule arithmetic: steps per epoch = windows / workers / batch size.
steps_per_epoch = max(1, math.ceil(n_seqs / pt["num_workers"] / pt["batch_size"]))
total_steps = steps_per_epoch * pt["epochs"]
warmup_steps = int(pt.get("warmup_ratio", 0.0) * total_steps)
print(f"{n_seqs:,} windows · global batch {pt['batch_size'] * pt['num_workers']} "
      f"· ~{steps_per_epoch} steps/epoch × {pt['epochs']} epochs = ~{total_steps:,} optimizer steps "
      f"(warmup {warmup_steps})")

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": vocab_path, "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "seed": 0,
        # Optimizer settings + warmup/cosine schedule, from the scale config.
        "weight_decay": pt.get("weight_decay", 0.0), "betas": tuple(pt.get("betas", (0.9, 0.999))),
        "lr_schedule": pt.get("lr_schedule"), "total_steps": total_steps,
        "warmup_steps": warmup_steps, "min_lr_ratio": pt.get("min_lr_ratio", 0.0),
    },
    # The one line that sets the scale: worker count, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name=f"transaction_fm_pretrain_{SCALE}",      # scale-specific -> no cross-scale resume clash
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy the weights to the path Part 5 reads
print(f"done — final lm_loss {result.metrics['lm_loss']:.3f}, "
      f"perplexity {result.metrics['perplexity']:.1f} -> {paths['checkpoint']}")

8 windows · global batch 8 · ~1 steps/epoch × 2 epochs = ~2 optimizer steps (warmup 0)
(RayTrainWorker pid=135736, ip=10.0.114.120) [pretrain] epoch 1/2  lm_loss=8.742  ppl=6262.4  lr=3.00e-04
(RayTrainWorker pid=135736, ip=10.0.114.120) [pretrain] epoch 2/2  lm_loss=8.668  ppl=5816.5  lr=3.00e-04
(RayTrainWorker pid=135736, ip=10.0.114.120) Checkpoint successfully created at: Checkpoint(filesystem=local, path=/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain_mini/checkpoint_2026-07-21_22-38-52.001372)
done — final lm_loss 8.668, perplexity 5816.5 -> /mnt/cluster_storage/transaction-fm/model/mini/


## Read the results

Two numbers summarize the run: the training loss and its exponent, **perplexity** — the effective number of tokens the model is choosing between at each position. Guessing uniformly over the 6,251-token vocabulary would sit at perplexity 6,251; the `full` pretrain ends near **1.7**, fewer than two effective choices per token. That low a number is less surprising than it looks: several of the twelve fields (customer id, card index, state) repeat on nearly every transaction, so the model learns them immediately, and the remaining uncertainty concentrates in the informative fields — amount, merchant, time.

At `mini` the output above shows what two optimizer steps buy: the run starts almost exactly at the ceiling (a random init lands near it, even slightly above) and barely moves. That is all `mini` is for. The number that matters is the downstream fraud comparison in Part 6, built on the `full` pretrain.

In [4]:
m = result.metrics
print(f"final causal-LM loss {m['lm_loss']:.3f}   ·   perplexity {m['perplexity']:.1f}")

final causal-LM loss 8.668   ·   perplexity 5816.5


## Save the model for the later parts

Parts 5 through 8 load this model with standard Hugging Face tooling (NVIDIA's embedding code calls `AutoModelForCausalLM.from_pretrained`), so we save the trained weights in that format on shared storage.

In [5]:
import torch
from src.model import build_model

# Rebuild the model from the checkpoint, then save its inner Hugging Face model.
ck = paths["checkpoint"]
with open(os.path.join(ck, "model_config.json")) as f:
    mc = json.load(f)
m = build_model(vocab_path=os.path.join(ck, "vocab.json"), arch=mc["arch"], max_len=mc["max_len"])
sd = torch.load(os.path.join(ck, "model.pt"), map_location="cpu")
sd = sd["model_state_dict"] if isinstance(sd, dict) and "model_state_dict" in sd else sd
missing, unexpected = m.load_state_dict(sd, strict=False)
os.makedirs(paths["hf"], exist_ok=True)
m.lm.save_pretrained(paths["hf"])
print(f"exported HF decoder -> {paths['hf']}  (state_dict: missing {len(missing)}, unexpected {len(unexpected)})")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

exported HF decoder -> /mnt/cluster_storage/transaction-fm/model_hf/mini/  (state_dict: missing 0, unexpected 0)


## Scaling factors

Training cost is arithmetic: tokens × epochs × model size sets the work, and workers divide the wall-clock. At `full` that is 64,335 windows × 4096 tokens × 8 epochs through the 29M-parameter model — ~16,000 optimizer steps, about two hours on 8×A10G. Each worker trains on its own shard and the gradients average after every step, so eight workers run an epoch in roughly an eighth of the time.

Memory is the constraint that mattered in practice. The loss needs a prediction for every position — a (batch × 4096 × 6,251) tensor — and at batch 8 per worker that tensor alone overflowed the A10G's 15.6 GiB. `full` runs batch 4 with gradient checkpointing, which keeps the peak near 9 GiB, and the fix lives in the scale config, not the training loop. If the model itself ever outgrows one GPU, `use_fsdp` splits it across workers — at 29M parameters it fits easily.

The GPU workers exist for the two hours of this stage and scale back to zero afterward.

## Takeaways

We trained the foundation model: next-token prediction over the token windows, checkpointed to shared storage, and saved in Hugging Face format for the later parts. The training function is plain PyTorch; Ray Train supplied the workers, the data sharding, the gradient averaging, and the checkpointing, and `ScalingConfig` is the only difference between one CPU worker at `mini` and eight GPUs at `full`.

This is also the one stage NVIDIA's repo skips: their notebook trains a 30-step demonstration and downloads the real weights. Our ~16,000-step run is the real training — and the Part 1 results table shows what it is worth.

---

## Next

**Part 5 — Extract embeddings**: run every transaction through the trained model to get one embedding per transaction — CPU workers feed batches to GPU workers.